# Quickly print results

In [ ]:
import glob
from piarena.utils import load_json

def print_all_results(name:str):
    result_paths = sorted(glob.glob(f"{name}"))

    for result_path in result_paths:
        try:
            result = load_json(result_path)
        except Exception as e:
            print(e)
            print(result_path)
            continue

        result_name = result_path.split("/")[-1]
        print(result_name.replace(".json", ""))

        print(f"Num of samples: {len(result)}")
        print(f"Utility: {round(sum([r['utility'] for r in result.values()]) / len(result), 2)}")
        print(f"ASR: {round(sum([r['asr'] for r in result.values()]) / len(result), 2)}")

        print("\n")

In [ ]:
# Enter the experiment name here
print_all_results("results/evaluation_results/[Experiment Name]/*.json")

dolly_closed_qa-Qwen3-combined-pisanitizer
Utility: 0.97
ASR: 0.04


dolly_closed_qa-Qwen3-direct-pisanitizer
Utility: 0.97
ASR: 0.04


dolly_closed_qa-Qwen3-none-pisanitizer
Utility: 0.99
ASR: 0.03


dolly_information_extraction-Qwen3-combined-pisanitizer
Utility: 0.94
ASR: 0.04


dolly_information_extraction-Qwen3-direct-pisanitizer
Utility: 0.93
ASR: 0.04


dolly_information_extraction-Qwen3-none-pisanitizer
Utility: 0.99
ASR: 0.03


dolly_summarization-Qwen3-combined-pisanitizer
Utility: 0.95
ASR: 0.05


dolly_summarization-Qwen3-direct-pisanitizer
Utility: 0.95
ASR: 0.04


dolly_summarization-Qwen3-none-pisanitizer
Utility: 0.99
ASR: 0.0


gov_report_long-Qwen3-combined-pisanitizer
Utility: 0.24
ASR: 0.02


gov_report_long-Qwen3-direct-pisanitizer
Utility: 0.24
ASR: 0.05


gov_report_long-Qwen3-none-pisanitizer
Utility: 0.24
ASR: 0.0


hotpotqa_long-Qwen3-combined-pisanitizer
Utility: 0.55
ASR: 0.0


hotpotqa_long-Qwen3-direct-pisanitizer
Utility: 0.56
ASR: 0.01


hotpotqa_long-Qw

# AgentDojo / AgentDyn Results

In [ ]:
import json
import os
from pathlib import Path
from collections import defaultdict


def load_agentdojo_results(
    logdir: str,
    pipeline_name: str,
    suites: list[str],
    attack: str | None = None,
):
    """Load agentdojo/agentdyn results from disk, tolerating incomplete runs.

    Args:
        logdir: Root results directory (e.g. "results/agent_evaluations/agentdojo/default").
        pipeline_name: Pipeline directory name (e.g. "GPT_4O", "meta-llama_Llama-3.1-8B-Instruct").
        suites: List of suite names to load (e.g. ["workspace", "slack", "shopping"]).
        attack: Attack name (e.g. "important_instructions") or None for benign-only.

    Returns:
        Dict mapping suite_name -> {
            "utility_results": {(user_task, inj_task): bool, ...},
            "security_results": {(user_task, inj_task): bool, ...},
        }
    """
    logdir = Path(logdir)
    results = {}

    for suite_name in suites:
        suite_dir = logdir / pipeline_name / suite_name
        if not suite_dir.exists():
            print(f"[Warning] Suite directory not found: {suite_dir}")
            continue

        utility_results = {}
        security_results = {}

        # Discover user tasks from directory listing
        user_task_dirs = sorted([
            d.name for d in suite_dir.iterdir()
            if d.is_dir() and d.name.startswith("user_task_")
        ])

        for user_task_id in user_task_dirs:
            # Load benign (no-attack) results
            benign_file = suite_dir / user_task_id / "none" / "none.json"
            if benign_file.exists():
                with open(benign_file) as f:
                    data = json.load(f)
                utility_results[(user_task_id, "")] = data["utility"]

            # Load attack results if requested
            if attack is not None:
                attack_dir = suite_dir / user_task_id / attack
                if attack_dir.exists():
                    for inj_file in sorted(attack_dir.glob("injection_task_*.json")):
                        with open(inj_file) as f:
                            data = json.load(f)
                        inj_task_id = inj_file.stem
                        utility_results[(user_task_id, inj_task_id)] = data["utility"]
                        security_results[(user_task_id, inj_task_id)] = data["security"]

        results[suite_name] = {
            "utility_results": utility_results,
            "security_results": security_results,
        }

    return results


def print_agentdojo_results(
    logdir: str,
    pipeline_name: str,
    suites: list[str],
    attack: str | None = None,
):
    """Load and print agentdojo/agentdyn results with per-suite and combined metrics.

    Handles incomplete runs by averaging over available samples.
    """
    results = load_agentdojo_results(logdir, pipeline_name, suites, attack)

    if not results:
        print("No results found.")
        return

    combined_utility = {}
    combined_security = {}

    print(f"Pipeline: {pipeline_name}")
    print(f"Attack:   {attack or 'none (benign)'}")
    print(f"Logdir:   {logdir}")
    print()

    for suite_name in suites:
        if suite_name not in results:
            continue

        suite = results[suite_name]
        util_res = suite["utility_results"]
        sec_res = suite["security_results"]

        # Count unique user tasks and injection tasks found
        user_tasks_benign = {ut for ut, it in util_res if it == ""}
        user_tasks_attack = {ut for ut, it in sec_res}
        inj_tasks_per_ut = defaultdict(set)
        for ut, it in sec_res:
            inj_tasks_per_ut[ut].add(it)

        print(f"{'='*55}")
        print(f"Suite: {suite_name}")
        print(f"{'='*55}")

        # Benign utility
        benign_vals = [v for (ut, it), v in util_res.items() if it == ""]
        if benign_vals:
            avg_util = sum(benign_vals) / len(benign_vals)
            print(f"  Benign utility:       {avg_util * 100:.2f}%  ({len(benign_vals)} user tasks)")
        else:
            print(f"  Benign utility:       N/A (no benign results)")

        if attack is not None:
            # Utility under attack (from attack runs)
            attack_util_vals = [v for (ut, it), v in util_res.items() if it != ""]
            if attack_util_vals:
                avg_attack_util = sum(attack_util_vals) / len(attack_util_vals)
                print(f"  Utility under attack: {avg_attack_util * 100:.2f}%  ({len(attack_util_vals)} task pairs)")

            # Security (1 - ASR)
            if sec_res:
                sec_vals = list(sec_res.values())
                avg_security = sum(sec_vals) / len(sec_vals)
                print(f"  Security:             {avg_security * 100:.2f}%  ({len(sec_vals)} task pairs)")
                print(f"  User tasks w/ attack: {len(user_tasks_attack)}")
            else:
                print(f"  Security:             N/A (no attack results)")

        print()

        # Add to combined results (prepend suite name)
        for (ut, it), v in util_res.items():
            combined_utility[(f"{suite_name}_{ut}", it)] = v
        for (ut, it), v in sec_res.items():
            combined_security[(f"{suite_name}_{ut}", it)] = v

    # Combined results across all suites
    if len(results) > 1:
        print(f"{'='*55}")
        print(f"COMBINED ({', '.join(results.keys())})")
        print(f"{'='*55}")

        benign_vals = [v for (ut, it), v in combined_utility.items() if it == ""]
        if benign_vals:
            print(f"  Benign utility:       {sum(benign_vals) / len(benign_vals) * 100:.2f}%  ({len(benign_vals)} user tasks)")

        if attack is not None:
            attack_util_vals = [v for (ut, it), v in combined_utility.items() if it != ""]
            if attack_util_vals:
                print(f"  Utility under attack: {sum(attack_util_vals) / len(attack_util_vals) * 100:.2f}%  ({len(attack_util_vals)} task pairs)")

            if combined_security:
                sec_vals = list(combined_security.values())
                avg_sec = sum(sec_vals) / len(sec_vals)
                print(f"  Security:             {avg_sec * 100:.2f}%  ({len(sec_vals)} task pairs)")
        print()

In [ ]:
# === Configure and run ===

logdir = "results/agent_evaluations/agentdojo/agentdojo"     # --name passed to main_agentdojo.py
pipeline_name = "GPT_4O"                                      # model name with "/" replaced by "_"

# AgentDojo suites: workspace, slack, travel, banking
# AgentDyn suites: shopping, github, dailylife
suites = ["workspace", "slack", "travel", "banking"]

# Set attack name or None for benign-only
attack = "important_instructions"

print_agentdojo_results(logdir, pipeline_name, suites, attack)